In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
import os
import shutil
import tarfile
import zipfile
from pathlib import Path
from google.colab import drive
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
#First mount Drive for the dataset if it is not mounted or session is restarted.
drive.mount('/content/drive')

In [ ]:
#Clone or pull the latest code
import os
REPO = "LaneDetection"
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/abdullahtapanci/LaneDetection.git /content/{REPO}
else:
    !cd /content/{REPO} && git pull

%cd /content/{REPO}

EXTRACT THE DATA INTO COLAB ENVIRONMENT
1.SETUP

In [ ]:
# =========================
# CULane local extraction setup (Colab)
# =========================

# 1) Configure paths
DRIVE_CULANE_DIR = Path("/content/drive/MyDrive/CULane_Dataset")
LOCAL_WORK_DIR   = Path("/content")
LOCAL_ARCHIVE_DIR = LOCAL_WORK_DIR / "culane_archives"
LOCAL_CULANE_DIR  = LOCAL_WORK_DIR / "CULane"

LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CULANE_DIR.mkdir(parents=True, exist_ok=True)

# 2) Which archives to process
# Include only files that exist in your Drive folder
ARCHIVES = [
    "driver_100_30frame.tar.gz",
    "driver_161_90frame.tar.gz",
    "driver_182_30frame.tar.gz",
    "driver_193_90frame.tar.gz",
    "driver_37_30frame.tar.gz",
    "annotations_new.tar.gz",
    "laneseg_label_w16.tar.gz",
    "laneseg_label_w16_test.zip",
    "list.tar.gz",
]

# 3) Copy from Drive -> local (skip if already copied with same size)
def copy_if_needed(src: Path, dst: Path):
    if not src.exists():
        print(f"[WARN] Missing on Drive: {src.name}")
        return False
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f"[SKIP] Already copied: {src.name}")
        return True
    print(f"[COPY] {src.name}")
    shutil.copy2(src, dst)
    return True

copied = []
for name in ARCHIVES:
    src = DRIVE_CULANE_DIR / name
    dst = LOCAL_ARCHIVE_DIR / name
    ok = copy_if_needed(src, dst)
    if ok:
        copied.append(dst)

print(f"\nTotal ready archives: {len(copied)}")

EXTRACTION PROCESS

In [ ]:
# =========================
# Extract archives into /content/CULane
# =========================
from pathlib import Path
import tarfile, zipfile

LOCAL_ARCHIVE_DIR = Path("/content/culane_archives")
LOCAL_CULANE_DIR  = Path("/content/CULane")
LOCAL_CULANE_DIR.mkdir(parents=True, exist_ok=True)

def extract_tar_gz(archive_path: Path, out_dir: Path):
    print(f"[EXTRACT tar] {archive_path.name}")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=out_dir)

def extract_zip(archive_path: Path, out_dir: Path):
    print(f"[EXTRACT zip] {archive_path.name}")
    with zipfile.ZipFile(archive_path, "r") as z:
        z.extractall(path=out_dir)

for p in sorted(LOCAL_ARCHIVE_DIR.iterdir()):
    if p.suffixes[-2:] == [".tar", ".gz"]:
        extract_tar_gz(p, LOCAL_CULANE_DIR)
    elif p.suffix == ".zip":
        extract_zip(p, LOCAL_CULANE_DIR)
    else:
        print(f"[SKIP] Unknown format: {p.name}")

print("\nExtraction complete.")

VERIFY

In [ ]:
# =========================
# Quick verification
# =========================
from pathlib import Path

root = Path("/content/CULane")

# count common file types
jpg_count = len(list(root.rglob("*.jpg")))
png_count = len(list(root.rglob("*.png")))
lines_count = len(list(root.rglob("*.lines.txt")))
list_files = list(root.rglob("*list*")) + list(root.rglob("train*.txt")) + list(root.rglob("val*.txt")) + list(root.rglob("test*.txt"))

print(f"Root: {root}")
print(f"JPG count      : {jpg_count}")
print(f"PNG count      : {png_count}")
print(f"lines.txt count: {lines_count}")
print(f"List-like files: {len(list_files)}")

# Show some list files for sanity
for f in sorted(set(list_files))[:20]:
    print(" -", f)

NOW, WE HAVE TO CREATE BINARY MASK FOLDER AND ALSO CONSTRUCT MANIFEST TXT FILES FOR TRAINING AND VALIDATION

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

CULANE_ROOT = Path("/content/CULane")

TRAIN_GT = CULANE_ROOT / "list" / "train_gt.txt"
VAL_GT   = CULANE_ROOT / "list" / "val_gt.txt"

BINARY_MASK_DIR = CULANE_ROOT / "binary_label"
BINARY_MASK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_MANIFEST = CULANE_ROOT / "train.txt"
VAL_MANIFEST   = CULANE_ROOT / "val.txt"

print("CULANE_ROOT:", CULANE_ROOT)
print("TRAIN_GT exists:", TRAIN_GT.exists(), TRAIN_GT)
print("VAL_GT exists:", VAL_GT.exists(), VAL_GT)
print("BINARY_MASK_DIR:", BINARY_MASK_DIR)


In [ ]:
def clean_rel_path(path_text):
    return path_text.strip().lstrip("/")

def resolve_path(root, rel_path):
    return root / clean_rel_path(rel_path)

def binary_mask_rel_from_seg_rel(seg_rel):
    seg_rel = clean_rel_path(seg_rel)

    # Example:
    # laneseg_label_w16/driver_100_30frame/...
    # -> binary_label/driver_100_30frame/...
    parts = Path(seg_rel).parts

    if parts[0] == "laneseg_label_w16":
        return str(Path("binary_label", *parts[1:]))

    # fallback if your extracted path is different
    return str(Path("binary_label") / Path(seg_rel).name)


In [ ]:
def create_binary_mask(seg_path, binary_path):
    seg = cv2.imread(str(seg_path), cv2.IMREAD_GRAYSCALE)

    if seg is None:
        raise FileNotFoundError(f"Could not read segmentation mask: {seg_path}")

    binary = (seg > 0).astype(np.uint8) * 255

    binary_path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(binary_path), binary)

    if not ok:
        raise RuntimeError(f"Could not write binary mask: {binary_path}")


In [ ]:
def build_lanedataset_manifest(gt_file, out_manifest, root, overwrite_binary=False):
    """
    Converts CULane gt file into your current LaneDataset format:
        image_path binary_mask_path instance_mask_path

    CULane gt rows usually look like:
        /driver_xxx/...jpg /laneseg_label_w16/driver_xxx/...png 1 1 0 0
    """

    assert gt_file.exists(), f"Missing gt file: {gt_file}"

    lines = [
        line.strip()
        for line in gt_file.read_text().splitlines()
        if line.strip()
    ]

    manifest_rows = []
    missing = []
    bad_rows = []

    for line_idx, line in enumerate(tqdm(lines, desc=f"Building {out_manifest.name}"), start=1):
        parts = line.split()

        if len(parts) < 2:
            bad_rows.append((line_idx, line))
            continue

        img_rel = clean_rel_path(parts[0])
        seg_rel = clean_rel_path(parts[1])

        img_path = resolve_path(root, img_rel)
        seg_path = resolve_path(root, seg_rel)

        binary_rel = binary_mask_rel_from_seg_rel(seg_rel)
        binary_path = resolve_path(root, binary_rel)

        if not img_path.exists():
            missing.append(("image", line_idx, img_path))
            continue

        if not seg_path.exists():
            missing.append(("segmentation", line_idx, seg_path))
            continue

        if overwrite_binary or not binary_path.exists():
            create_binary_mask(seg_path, binary_path)

        # Current LaneDataset expects:
        # image, binary mask, instance mask
        manifest_rows.append(f"{img_rel} {binary_rel} {seg_rel}")

    if bad_rows:
        print("Bad rows:")
        for item in bad_rows[:20]:
            print(item)
        raise RuntimeError(f"Found {len(bad_rows)} bad rows in {gt_file}")

    if missing:
        print("Missing files:")
        for kind, line_idx, path in missing[:20]:
            print(f"[{kind}] line {line_idx}: {path}")
        raise RuntimeError(f"Found {len(missing)} missing files in {gt_file}")

    out_manifest.write_text("\n".join(manifest_rows) + "\n")

    print("Wrote:", out_manifest)
    print("Rows:", len(manifest_rows))


In [ ]:
build_lanedataset_manifest(
    gt_file=TRAIN_GT,
    out_manifest=TRAIN_MANIFEST,
    root=CULANE_ROOT,
    overwrite_binary=False,
)

build_lanedataset_manifest(
    gt_file=VAL_GT,
    out_manifest=VAL_MANIFEST,
    root=CULANE_ROOT,
    overwrite_binary=False,
)


VERIFY GENERATED MANIFEST FILES

In [ ]:
def verify_manifest(manifest_path, root, sample_n=20):
    lines = [
        line.strip()
        for line in manifest_path.read_text().splitlines()
        if line.strip()
    ]

    print("Manifest:", manifest_path)
    print("Rows:", len(lines))

    for line in lines[:sample_n]:
        parts = line.split()

        assert len(parts) == 3, f"Expected 3 columns, got {len(parts)}: {line}"

        img_rel, bin_rel, inst_rel = parts

        img_path = resolve_path(root, img_rel)
        bin_path = resolve_path(root, bin_rel)
        inst_path = resolve_path(root, inst_rel)

        assert img_path.exists(), f"Missing image: {img_path}"
        assert bin_path.exists(), f"Missing binary mask: {bin_path}"
        assert inst_path.exists(), f"Missing instance mask: {inst_path}"

        img = cv2.imread(str(img_path))
        bin_mask = cv2.imread(str(bin_path), cv2.IMREAD_GRAYSCALE)
        inst_mask = cv2.imread(str(inst_path), cv2.IMREAD_GRAYSCALE)

        assert img is not None, f"Could not read image: {img_path}"
        assert bin_mask is not None, f"Could not read binary mask: {bin_path}"
        assert inst_mask is not None, f"Could not read instance mask: {inst_path}"

        unique_bin = set(np.unique(bin_mask).tolist())
        assert unique_bin.issubset({0, 255}), f"Binary mask is not binary: {bin_path}, values={unique_bin}"

    print("Passed:", manifest_path)

verify_manifest(TRAIN_MANIFEST, CULANE_ROOT)
verify_manifest(VAL_MANIFEST, CULANE_ROOT)


CHECK IF IT WORKS WITH THE EXISTING LANE DATASET FUNCTION

In [ ]:
import sys
sys.path.append("/content/LaneDetection")

import torch
from src.data.dataset import LaneDataset
from src.data.transforms import transform_image

train_ds = LaneDataset(
    manifest_path=str(TRAIN_MANIFEST),
    root_dir=str(CULANE_ROOT),
    transform=transform_image,
    training=False,
)

val_ds = LaneDataset(
    manifest_path=str(VAL_MANIFEST),
    root_dir=str(CULANE_ROOT),
    transform=transform_image,
    training=False,
)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

for name, ds in [("train", train_ds), ("val", val_ds)]:
    img, bin_mask, inst_mask = ds[0]

    print(f"\n{name}")
    print("img:", img.shape, img.dtype, img.min().item(), img.max().item())
    print("bin_mask:", bin_mask.shape, bin_mask.dtype, torch.unique(bin_mask))
    print("inst_mask:", inst_mask.shape, inst_mask.dtype, torch.unique(inst_mask)[:20])

    assert img.shape[0] == 3
    assert bin_mask.shape[0] == 1
    assert inst_mask.shape[0] == 1
    assert set(torch.unique(bin_mask).tolist()).issubset({0.0, 1.0})

print("\nLaneDataset check passed.")
